# 03 — Integração e limpeza

**Objetivo:** juntar as três bases e produzir os dois datasets que as análises
usam.

**Entradas:**
- `censo_domicilios_sp.csv` (notebook 01 — IBGE Censo 2022)
- `internacoes_sp.csv` (notebook 02 — SIH/SUS por residência, por ano)
- `data/external/idh_sp.csv` (Atlas Brasil — variável de controle)

**Saídas:**
- `dataset_consolidado_sp.csv` — **painel** município × ano (645 × 5 = 3.225
  linhas). Serve para os gráficos de evolução temporal.
- `dataset_municipios_sp_bruto.csv` — 645 municípios, totais do período, sem filtros
- `dataset_municipios_sp.csv` — versão **limpa**, usada para testar a hipótese

**Por que a hipótese é testada na base por município, e não no painel:**
`pct_idosos_sozinhos` vem do Censo 2022 e é **constante ao longo dos 5 anos**.
Rodar a regressão no painel repetiria cada município 5 vezes sem acrescentar
nenhuma informação nova sobre a hipótese, inflando o "n" de 645 para 3.225 e
os níveis de significância junto
([pseudorreplicação](https://en.wikipedia.org/wiki/Pseudoreplication)).


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np

censo = pd.read_csv(config.DATA_PROCESSED / "censo_domicilios_sp.csv")
internacoes = pd.read_csv(config.DATA_PROCESSED / "internacoes_sp.csv")

print("censo:", censo.shape, "| internacoes:", internacoes.shape)


## 3.1 IDH municipal (variável de controle)

O IDH entra como controle porque municípios mais pobres tendem a ter mais
internações por razões que não têm nada a ver com morar sozinho. Sem esse
controle, qualquer associação encontrada poderia ser só reflexo da
desigualdade socioeconômica entre municípios.

⚠️ O IDHM disponível é o de 2010 (o Censo 2022 ainda não tem IDHM oficial
publicado) e cobre 259 dos 645 municípios — declarar as duas coisas como
limitação no artigo.


In [ ]:
idh = pd.read_csv(config.DATA_EXTERNAL / "idh_sp.csv")
idh["municipio_norm"] = idh["municipio"].apply(config.normalizar_municipio)
idh = idh[["municipio_norm", "idh"]].rename(columns={"idh": "idhm"}).drop_duplicates("municipio_norm")

print(f"{len(idh)} municípios no arquivo de IDH")
print(f"{idh['municipio_norm'].isin(censo['municipio_norm']).sum()} deles batem com o Censo")


## 3.2 Painel município × ano

O merge parte da grade completa (645 municípios × 5 anos), não das
internações — assim, município/ano/capítulo sem nenhuma internação entra com
0 em vez de sumir da base.


In [ ]:
causas = list(config.CAUSAS_SIH.keys())

grade = censo[["municipio", "municipio_norm"]].merge(
    pd.DataFrame({"ano": config.ANOS_SIH}), how="cross"
)

internacoes_wide = internacoes.pivot_table(
    index=["municipio_norm", "ano"], columns="causa", values="internacoes", aggfunc="sum"
).reset_index()

painel = (
    grade
    .merge(internacoes_wide, on=["municipio_norm", "ano"], how="left")
    .merge(censo.drop(columns="municipio"), on="municipio_norm", how="left")
    .merge(idh, on="municipio_norm", how="left")
)
painel[causas] = painel[causas].fillna(0).astype(int)

painel["internacoes_total"] = painel[causas].sum(axis=1)
painel["taxa_internacao_100k_domicilios_idosos"] = (
    painel["internacoes_total"] / painel["domicilios_resp_idoso"] * 100_000
).round(2)

painel.to_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv", index=False)
print(painel.shape, "->  dataset_consolidado_sp.csv")
print(f"Internações no painel: {painel['internacoes_total'].sum():,} (deve bater com o notebook 02)")
painel.head()


## 3.3 Base por município (totais do período)

Soma os 5 anos. É nesta tabela que a hipótese é testada.


In [ ]:
mun = (
    painel
    .groupby(["municipio", "municipio_norm"], as_index=False)
    .agg({**{c: "sum" for c in causas},
          "internacoes_total": "sum",
          "domicilios_total": "first",
          "domicilios_unipessoais_total": "first",
          "domicilios_resp_idoso": "first",
          "idosos_sozinhos": "first",
          "pct_idosos_sozinhos": "first",
          "idhm": "first"})
)
mun["taxa_internacao_100k_domicilios_idosos"] = (
    mun["internacoes_total"] / mun["domicilios_resp_idoso"] * 100_000
).round(2)

print(mun.shape)
print(f"Municípios sem IDH: {mun['idhm'].isna().sum()}")
mun.head()


## 3.4 Limpeza

Três tratamentos, nesta ordem:

1. **Arredondamento** — taxas com 2 casas decimais (feito acima). Sem perda de
   informação real: a precisão adicional era ruído de divisão.
2. **Outliers** — removidos pela regra de Tukey (taxa acima de Q3 + 1,5×IQR).
   Agora que o dado é por residência, os outliers restantes são
   predominantemente municípios **muito pequenos**, onde o denominador baixo
   deixa a taxa por 100 mil instável — variância estatística legítima, não o
   viés sistemático que tínhamos com o dado por local de internação.
3. **Municípios sem IDH** — eliminados em vez de imputados. Cerca de 60% da
   coluna está ausente e não há aqui fonte confiável para imputar com critério;
   inventar valor para a variável de controle seria pior do que reduzir a
   amostra.

A base bruta fica salva à parte para a checagem de robustez do notebook 04.


In [ ]:
q1, q3 = mun["taxa_internacao_100k_domicilios_idosos"].quantile([0.25, 0.75])
limite_tukey = q3 + 1.5 * (q3 - q1)
mun["outlier_taxa"] = mun["taxa_internacao_100k_domicilios_idosos"] > limite_tukey

print(f"Limite de Tukey (Q3 + 1,5xIQR): {limite_tukey:,.0f} internações por 100 mil")
print(f"Outliers detectados: {mun['outlier_taxa'].sum()}")
print()
print(mun[mun["outlier_taxa"]][["municipio", "domicilios_resp_idoso", "internacoes_total",
                                 "taxa_internacao_100k_domicilios_idosos"]]
      .sort_values("taxa_internacao_100k_domicilios_idosos", ascending=False)
      .to_string(index=False))


In [ ]:
mun.to_csv(config.DATA_PROCESSED / "dataset_municipios_sp_bruto.csv", index=False)

limpo = mun[~mun["outlier_taxa"] & mun["idhm"].notna()].drop(columns="outlier_taxa")
limpo.to_csv(config.DATA_PROCESSED / "dataset_municipios_sp.csv", index=False)

print(f"Bruto: {len(mun)} municípios  -> dataset_municipios_sp_bruto.csv")
print(f"Limpo: {len(limpo)} municípios -> dataset_municipios_sp.csv")
print(f"  removidos por outlier: {mun['outlier_taxa'].sum()}")
print(f"  removidos por falta de IDH: {mun['idhm'].isna().sum()}")


## 3.5 Conferência — Rio Claro

Rio Claro é o município do estudo de caso (notebook 05), então vale conferir
que sobreviveu à limpeza e que os números fazem sentido.


In [ ]:
rc = limpo[limpo["municipio"] == config.RIO_CLARO_NOME]
if rc.empty:
    print(f"ATENÇÃO: {config.RIO_CLARO_NOME} caiu fora da base limpa -- confira o bruto.")
    rc = mun[mun["municipio"] == config.RIO_CLARO_NOME]
print(rc.T.to_string())
